# Chess-Coach Agent — LoRA on **Google Colab** (free T4)

Same QLoRA run as the Kaggle notebook, adapted for Colab free tier. Trains a
**Gemma 4 E2B** LoRA on the v1.2 corpus, saves the adapter, you serve it locally
as q4_0 GGUF on your RTX 4060.

## Free-T4 reality (read this)
Free Colab idle-disconnects after ~90 min of **no browser activity** and has a
dynamic session ceiling. So:
- `MAX_STEPS = 300` (~5.4h at ~65s/update) to fit ONE sitting. Keep the tab open + the machine awake.
- Checkpoints stream to **Google Drive** (`USE_DRIVE=True`): the `best/` adapter is
  re-saved every `EVAL_EVERY` updates, so a disconnect still leaves a salvageable adapter on Drive.
- On a faster runtime (L4/A100) you can raise `MAX_STEPS` — Cell 2 prints the projected wall-time.

## Setup (do first)
1. **Runtime → Change runtime type → T4 GPU**.
2. **🔑 Secrets** (left sidebar) → add `HF_TOKEN` (HF token, read scope), toggle **Notebook access** on. Accept the Gemma license once: https://huggingface.co/google/gemma-4-E2B-it-qat-q4_0-unquantized
3. Run cells top to bottom. Cell 7 will prompt to mount Drive.

In [ ]:
# Cell 1 — config
import os

MODEL = "gemma4_e2b"  # E2B fits one T4 in 4-bit. E4B untested on T4.
HF_REPO = {
    "gemma4_e4b": "google/gemma-4-E4B-it-qat-q4_0-unquantized",
    "gemma4_e2b": "google/gemma-4-E2B-it-qat-q4_0-unquantized",
}[MODEL]
NO_4BIT = False  # keep 4-bit QLoRA on a T4 (~2.5GB, matches the q4_0 GGUF you serve)

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/content/llm-and-engine"
OUTPUT = "gemma4_chess_colab"

# Persist checkpoints to Drive so a disconnect doesn't lose the run.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/chess_coach_runs"

# free-T4 sizing: ~65s/update; 300*8 microbatch ~= 5.4h, fits one session.
# best/ adapter is re-saved every EVAL_EVERY updates -> salvage points on Drive.
MAX_STEPS = 300
RANK = 16
TARGETS = "attn-only"
GRAD_ACCUM = 8
MAX_SEQ = 1280
EVAL_EVERY = 100
MAX_VAL = 128
print("config:", MODEL, "steps=", MAX_STEPS, "drive=", USE_DRIVE)

In [ ]:
# Cell 2 — GPU preflight + wall-time projection
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
per = {"A100": 14, "L4": 32, "T4": 65}
sec = next((v for k, v in per.items() if k in name), 65)
print(f"device={name} | ~{sec}s/update -> {MAX_STEPS} steps ~= {MAX_STEPS*sec/3600:.1f}h "
      f"(raise MAX_STEPS if this is well under your session limit)")

In [ ]:
# Cell 3 — clone repo (code + data; skip LFS weights)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 — deps (Gemma 4 needs recent transformers; drop torchao like Kaggle)
import subprocess, sys
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "trl>=0.13",
    "bitsandbytes>=0.45", "datasets", "sentencepiece", "protobuf", "python-chess")
# Colab may ship torchao; recent PEFT raises on old versions inside get_peft_model.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bnb", bitsandbytes.__version__)

In [ ]:
# Cell 5 — HF login (Colab Secrets) + download base model
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN in the Colab Secrets panel (key icon) and enable notebook access."
login(token)
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))

In [ ]:
# Cell 6 — data check (REGENERATED qualitative corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - pull the branch again"

In [ ]:
# Cell 7 — (optional) mount Drive so checkpoints persist, then train
import subprocess, sys, os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    runs = f"{WORKDIR}/runs"
    # symlink runs/ -> Drive so the trainer's best/ + final adapter land on Drive LIVE
    if not os.path.islink(runs) and not os.path.exists(runs):
        os.symlink(DRIVE_DIR, runs)
    print("checkpoints ->", DRIVE_DIR)

cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--eval-every", str(EVAL_EVERY), "--max-val", str(MAX_VAL),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})

In [ ]:
# Cell 8 — zip the adapter + download (also already on Drive if USE_DRIVE)
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/content/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("zip at", out_zip + ".zip")
try:
    from google.colab import files
    files.download(out_zip + ".zip")
except Exception as e:
    print("download from the Files panel instead:", out_zip + ".zip", "|", e)